# final_01 台区负荷日前预测全流程（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`../data/load_curve.csv`（2 台区 × 8760 h，含重复行 / 缺失 / 尖峰 / 负值 / 哨兵 9999）。
>
> **每题都跟了 `# 提示：`**，照着提示能独立做完。
> 如果 assert 的数字对不上，**先回头看上一步的输出**，别急着改 assert —— 那些数字都是实跑出来的。

## 环境提示（本章必读）

`lightgbm` 与 `torch` 在本机各带一套 OpenMP（`libomp` / `libiomp`），
同时开多线程会互相踩内存直接崩掉进程，`try/except` 都抓不住。
`IMPORTS` 已经**在任何 import 之前**把 `OMP_NUM_THREADS` 钉成 `1`，
**不要删掉那两行**，否则任务 10（LSTM）会莫名段错误。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 分组阈值清洗 | 4 | §一 |
| 2 | 特征工程与周期编码 | 2 | §二 |
| 3 | 按时间切分 + 泄漏自检 | 7 | §三 |
| 4 | 四条朴素基线 | 2 | §四 |
| 5 | Ridge 与 LightGBM | 2 | §五 |
| 6 | 误差的时段结构 | 2 | §六 |
| 7 | 直接法多步预测 | 4 | §七 |
| 8 | 递归法与 teacher forcing | 3 | §八 |
| 9 | 归因拆解 | 1 | §九 |
| 10 | LSTM 对照实验 | 2 | §十 |
| 11 | 残差白噪声检验 | 1 | §十一 |
| 12 | 画三张图 | 3 | §十二 |
| 13 | 自动生成结题报告 | 1 | §十三 |
| 14 | 跨台区泛化 | 3 | §十四 |

> **建议顺序**：1 → 2 → 3 必须先做完，后面全部依赖它们产出的 `y / t / FEAT / tr / te`。
> 第 7、8 两题依赖第 3 题的 `cut`；第 10 题依赖第 7 题的 `opos`。

In [ ]:
import os

# ⚠️ 本机 lightgbm 用 libomp、torch 用 libiomp，两套 OpenMP 同时开多线程会互相踩内存，
# 直接 SIGSEGV（try/except 抓不住、报错也看不到）。必须在任何 import 之前把线程数钉死。
os.environ["OMP_NUM_THREADS"] = "1"

import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox

DATA = Path("../data")          # notebook 的 cwd = coding/05_timeseries/final
CSV = DATA / "load_curve.csv"

H = 24                          # 预测视野：未来 24 小时（日前预测）
SEED = 42
STATION = "STATION_A_01"        # 居民型：晚高峰 + 早次峰
OTHER = "STATION_B_02"          # 工业型：日间平台 + 周末腰斩
N_HEAD = 168                    # 滑窗最长 168 小时，特征表头部会有 168 行 NaN

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
print("numpy", np.__version__, "| pandas", pd.__version__, "| lightgbm", lgb.__version__)

In [ ]:
# --------------------------------------------------------------------------- #
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容
# --------------------------------------------------------------------------- #

HOLIDAY_RANGES = [
    ("2025-01-01", "2025-01-01"),   # 元旦
    ("2025-01-28", "2025-02-04"),   # 春节
    ("2025-04-04", "2025-04-06"),   # 清明
    ("2025-05-01", "2025-05-05"),   # 劳动节
    ("2025-05-31", "2025-06-02"),   # 端午
    ("2025-10-01", "2025-10-08"),   # 中秋 + 国庆
]
HOLIDAYS = set()
for _lo, _hi in HOLIDAY_RANGES:
    HOLIDAYS |= set(pd.date_range(_lo, _hi, freq="D").strftime("%Y-%m-%d"))

# 多步预测用的「精简特征集」：只保留「目标时刻前一小时就能算出来」的量
RED_COLS = ["h_sin", "h_cos", "d_sin", "d_cos", "m_sin", "m_cos",
            "is_weekend", "is_holiday", "T", "T_dev", "T2",
            "lag24", "lag48", "lag168", "ma24", "ma168", "sd24"]


def cal_block(idx, prefix=""):
    """把一组时间戳变成日历特征。

    `prefix` 用来给「目标时刻」的日历列加前缀：原点 t 的日历列叫 `h_sin`，
    目标 t+h 的日历列叫 `tgt_h_sin`。不加前缀直接拼表会**列名重名**，
    LightGBM 会直接抛 `Feature (h_sin) appears more than one time`。
    """
    d = {
        "h_sin": np.sin(2 * np.pi * idx.hour.to_numpy(float) / 24),
        "h_cos": np.cos(2 * np.pi * idx.hour.to_numpy(float) / 24),
        "d_sin": np.sin(2 * np.pi * idx.dayofweek.to_numpy(float) / 7),
        "d_cos": np.cos(2 * np.pi * idx.dayofweek.to_numpy(float) / 7),
        "m_sin": np.sin(2 * np.pi * idx.month.to_numpy(float) / 12),
        "m_cos": np.cos(2 * np.pi * idx.month.to_numpy(float) / 12),
        "is_weekend": (idx.dayofweek >= 5).astype(int),
        "is_holiday": idx.strftime("%Y-%m-%d").isin(HOLIDAYS).astype(int),
    }
    return pd.DataFrame({prefix + k: v for k, v in d.items()}, index=idx)


def clean_by_hour(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值。

    为什么不能用「全局中位数 × 3」：工业台区是「夜间低 + 日间高」的双峰分布，
    全局阈值会把整段日间峰误判成异常。居民台区单峰，用全局阈值看不出问题 ——
    换到 B 台区就露馅。
    """
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


def make_features(y, t):
    """日历 + 温度 + 滞后 + 滑窗。滑窗一律「先 shift(1) 再 rolling」，防泄漏。"""
    idx = y.index
    f = cal_block(idx)
    f["T"] = t.to_numpy(float)
    f["T_dev"] = np.abs(t.to_numpy(float) - 22.0)
    f["T2"] = t.to_numpy(float) ** 2
    f["y"] = y.to_numpy(float)
    for k in (1, 2, 3, 24, 25, 48, 168):
        f[f"lag{k}"] = y.shift(k).to_numpy(float)
    base = y.shift(1)
    for w in (3, 24, 168):
        f[f"ma{w}"] = base.rolling(w).mean().to_numpy(float)
        f[f"sd{w}"] = base.rolling(w).std().to_numpy(float)
    return f


def fit_lgb(X, y, n_estimators=300, learning_rate=0.06, num_leaves=31):
    """全流程统一超参的 LightGBM 封装：固定 seed、单线程，保证可复现。"""
    m = lgb.LGBMRegressor(n_estimators=n_estimators, learning_rate=learning_rate,
                          num_leaves=num_leaves, min_child_samples=20,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          n_jobs=1, random_state=SEED, verbose=-1)
    m.fit(X, y)
    return m


def metrics(y_true, y_pred):
    """MAE / RMSE / MAPE(%) / R2 —— 四个一起返回，竞赛报告要的就是这一组。"""
    a = np.asarray(y_true, dtype=float)
    b = np.asarray(y_pred, dtype=float)
    return (float(mean_absolute_error(a, b)),
            float(np.sqrt(mean_squared_error(a, b))),
            float(np.mean(np.abs((a - b) / a)) * 100),
            float(r2_score(a, b)))


def metric_row(name, m):
    """拼成排行榜的一行。"""
    return dict(模型=name, MAE=round(m[0], 4), RMSE=round(m[1], 4),
                MAPE=round(m[2], 4), R2=round(m[3], 6))


def seg_mean_row(seg, name, lo, hi):
    """把「按小时索引的误差表」在 [lo, hi] 这一段上取均值，拼成一行。"""
    return dict(时段=name,
                gbm=round(float(seg.loc[lo:hi, "gbm"].mean()), 4),
                lag24=round(float(seg.loc[lo:hi, "lag24"].mean()), 4),
                lag168=round(float(seg.loc[lo:hi, "lag168"].mean()), 4))


def red_row(tgt_pos, hist, cal_tgt, temp):
    """多步预测的一行精简特征。

    tgt_pos : 目标时刻在全序列里的整数位置
    hist    : 「截止到目标时刻**前 1 小时**」的完整负荷序列（允许含预测值）
    cal_tgt : 目标时刻的日历表（列名带 tgt_ 前缀）
    temp    : 温度全序列

    注意 `lag24 = hist[-24]`：因为 hist 的**最后一个元素**就是「目标前一小时」。
    写成 `hist[-25]` 就整体晚了一格 —— 这个 off-by-one 会让 MAE 从 13 涨到 21，
    且不报任何错。
    """
    c = cal_tgt.iloc[tgt_pos]
    return [c["tgt_h_sin"], c["tgt_h_cos"], c["tgt_d_sin"], c["tgt_d_cos"],
            c["tgt_m_sin"], c["tgt_m_cos"], c["tgt_is_weekend"], c["tgt_is_holiday"],
            temp[tgt_pos], abs(temp[tgt_pos] - 22.0), temp[tgt_pos] ** 2,
            hist[-24], hist[-48], hist[-168],
            float(np.mean(hist[-24:])), float(np.mean(hist[-168:])),
            float(np.std(hist[-24:]))]


def reduced_matrix(starts, yv, cal_tgt, temp):
    """精简特征矩阵：每个起点 p 一行，历史是 `yv[:p]`（截止到 p-1，目标为 y[p]）。"""
    return pd.DataFrame([red_row(p, yv[:p], cal_tgt, temp) for p in starts],
                        columns=RED_COLS)


def teacher_forcing_rows(opos, yv, cal_tgt, temp):
    """teacher forcing 的特征矩阵：每个发布点 × 每个 h，历史一律用**真值**。

    这不是可上线的做法（用到了未来真值），它的唯一用途是当**上界诊断**：
    「假如每一步喂进去的历史都是准的，这套精简特征最好能到多少」。
    """
    return pd.DataFrame(
        [red_row(p + h, yv[:p + h], cal_tgt, temp) for p in opos for h in range(1, H + 1)],
        columns=RED_COLS)


def train_lstm(yv, tv, h_sin, h_cos, cut_full, opos, mu, sd, tmu, tsd,
               seq_len=168, hidden=48, epochs=20, lr=3e-3):
    """四通道序列 [负荷, 温度, sin(h), cos(h)] → 单层 LSTM → 一次输出未来 24 步。

    mu/sd/tmu/tsd 是归一化参数。传 `0/1` 就是「不归一化」，传训练段的均值/标准差
    就是「归一化」—— 这是本章唯一要调的四个数，其余代码一个字都不用改。

    返回 (预测矩阵, 首个 epoch 的 loss, 末个 epoch 的 loss, 参数量)。
    """
    import torch
    import torch.nn as nn

    mat = np.stack([(yv - mu) / sd, (tv - tmu) / tsd, h_sin, h_cos], axis=1).astype(np.float32)
    starts = np.array([p for p in range(seq_len, cut_full) if p + H <= cut_full])
    x_tr = np.stack([mat[p - seq_len:p] for p in starts])
    y_tr = ((np.stack([yv[p + 1:p + 1 + H] for p in starts]) - mu) / sd).astype(np.float32)
    x_te = np.stack([mat[p - seq_len:p] for p in opos])

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn = nn.LSTM(4, hidden, batch_first=True)
            self.head = nn.Linear(hidden, H)

        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(out[:, -1])

    torch.manual_seed(SEED)
    dev = "mps" if torch.backends.mps.is_available() else "cpu"
    net = Net().to(dev)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = nn.SmoothL1Loss()
    xt = torch.tensor(x_tr, device=dev)
    yt = torch.tensor(y_tr, device=dev)
    hist = []
    for _ in range(epochs):
        perm = torch.randperm(len(xt))
        total = 0.0
        for b in range(0, len(xt), 128):
            j = perm[b:b + 128]
            opt.zero_grad()
            loss = lossf(net(xt[j]), yt[j])
            loss.backward()
            opt.step()
            total += loss.item() * len(j)
        hist.append(total / len(xt))
    with torch.no_grad():
        pred = net(torch.tensor(x_te, device=dev)).cpu().numpy() * sd + mu
    return pred, hist[0], hist[-1], sum(q.numel() for q in net.parameters())

In [ ]:
raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
print("原始行数", len(raw), "| 时间戳单调递增？", raw["时间戳"].is_monotonic_increasing)
print("台区", sorted(raw["台区编号"].unique()))

# 去重 → 排序。**顺序不能反**：先 asfreq 再排序，规则时间轴会被重复时刻搞乱
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
print("去重后", len(df), "行 | 重复行", len(raw) - len(df))

one = df[df["台区编号"] == STATION].set_index("时间戳").sort_index()
s_raw = one["负荷值"].asfreq("h")          # 未清洗的规则序列（这一步用不到温度）
t_raw = one["温度"].asfreq("h")
print("规则化后", len(s_raw), "小时 | 负荷缺失", int(s_raw.isna().sum()),
      "| 温度缺失", int(t_raw.isna().sum()))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

## 任务 1：分组阈值清洗（§一）

In [ ]:
med = s_raw.groupby(s_raw.index.hour).transform("median")
bad = ~s_raw.between(0, 3 * med) & s_raw.notna()
print("越界点数", int(bad.sum()), "| 其中负值", int((s_raw < 0).sum()))
print("越界原值", sorted(np.round(s_raw[bad].to_numpy(), 2).tolist()))

y = clean_by_hour(s_raw)
t = t_raw.interpolate(limit_direction="both")
print("清洗后缺失", int(y.isna().sum()), "| 被改动的点数", int((y - s_raw).abs().gt(1e-9).sum()))
print("清洗前 mean/std %.4f/%.4f → 清洗后 %.4f/%.4f"
      % (s_raw.mean(), s_raw.std(), y.mean(), y.std()))
print("最大绝对修正量 %.2f kW" % float((y - s_raw).abs().max()))

assert int(bad.sum()) == 7
assert int((s_raw < 0).sum()) == 2
assert int(y.isna().sum()) == 0 and int(t.isna().sum()) == 0
assert int((y - s_raw).abs().gt(1e-9).sum()) == 7
assert abs(float(y.std()) - 72.1933) < 1e-3
assert abs(float((y - s_raw).abs().max()) - 9480.56) < 0.01

## 任务 2：特征工程与周期编码（§二）

In [ ]:
FEAT = make_features(y, t)
print("形状", FEAT.shape, "| 列名", list(FEAT.columns))
print("含 NaN 的行", int(FEAT.isna().any(axis=1).sum()))
print("节假日小时", int(FEAT["is_holiday"].sum()), "| 周末小时", int(FEAT["is_weekend"].sum()))
print("温度线性相关 %.6f | |T-22| 相关 %.6f"
      % (np.corrcoef(t, y)[0, 1], np.corrcoef(np.abs(t - 22.0), y)[0, 1]))

# 周期的「整数距离」是骗人的：23 点与 0 点相差 23，实际只差 1 小时。
# 把它们编码成单位圆上的 (sin, cos) 后，两点间的**弦长**才反映真实距离。
p23 = pd.Timestamp("2025-03-05 23:00")
p00 = pd.Timestamp("2025-03-06 00:00")

chord_2300 = float(np.hypot(FEAT.loc[p23, "h_sin"] - FEAT.loc[p00, "h_sin"],
                            FEAT.loc[p23, "h_cos"] - FEAT.loc[p00, "h_cos"]))
print("23↔0 编码弦长 %.6f（不编码时是 23）" % chord_2300)

assert FEAT.shape == (8760, 25)
assert int(FEAT.isna().any(axis=1).sum()) == 168
assert int(FEAT["is_holiday"].sum()) == 672
assert int(FEAT["is_weekend"].sum()) == 2496
assert abs(float(np.corrcoef(np.abs(t - 22.0), y)[0, 1]) - 0.406437) < 1e-5
assert abs(chord_2300 - 0.261052) < 1e-6
# FEAT 还没 dropna，所以第 500 行就是全序列第 500 小时 —— 它的 lag168 应等于第 332 小时的负荷
assert np.isclose(FEAT["lag168"].iloc[500], y.iloc[500 - N_HEAD])

## 任务 3：按时间切分 + 泄漏自检（§三）

In [ ]:
OK = FEAT.dropna()

COLS = [c for c in OK.columns if c != "y"]
n = len(OK)
cut = int(n * 0.8)
tr, te = OK.iloc[:cut], OK.iloc[cut:]
print("样本 %d | 训练 %d | 测试 %d | 特征 %d 列" % (n, len(tr), len(te), len(COLS)))
print("训练", tr.index.min(), "~", tr.index.max())
print("测试", te.index.min(), "~", te.index.max())
print("训练 y mean/std %.4f/%.4f | 测试 %.4f/%.4f"
      % (tr["y"].mean(), tr["y"].std(), te["y"].mean(), te["y"].std()))

# ---- 泄漏自检：滑窗统计量必须只用到「目标前一小时及以前」----
# OK 第 i_check 行对应全序列位置 i_check + 168。rolling 默认右对齐：位置 j 的
# ma24 窗口是 j-23..j，而 base 已经 shift(1)，所以它实际覆盖 y[j-24:j]。
i_check = 300
j_check = i_check + N_HEAD

ma24_manual = float(y.iloc[j_check - 24:j_check].mean())
sd24_manual = float(y.iloc[j_check - 24:j_check].std(ddof=1))
lag1_manual = float(y.iloc[j_check - 1])
print("ma24  表内 %.4f | 手工 %.4f" % (OK["ma24"].iloc[i_check], ma24_manual))
print("sd24  表内 %.4f | 手工 %.4f" % (OK["sd24"].iloc[i_check], sd24_manual))
print("lag1  表内 %.4f | 手工 %.4f" % (OK["lag1"].iloc[i_check], lag1_manual))

assert (n, len(tr), len(te), len(COLS)) == (8592, 6873, 1719, 24)
assert tr.index.max() < te.index.min()          # 训练段与测试段完全不重叠
assert np.isclose(OK["ma24"].iloc[i_check], ma24_manual)
assert np.isclose(OK["sd24"].iloc[i_check], sd24_manual)
assert np.isclose(OK["lag1"].iloc[i_check], lag1_manual)
assert abs(float(te["y"].mean()) - 483.9447) < 1e-3

## 任务 4：四条朴素基线（§四）

In [ ]:
# 水平比：最近 24 小时均值 ÷ 它之前的 24 小时均值 —— 把周周期基线的整体水位拉回来
lvl = (te["ma24"] / te["ma24"].shift(24).bfill()).fillna(1.0)

base_rows = []
base_rows.append(metric_row("① 持续性 lag1", metrics(te["y"], te["lag1"])))
base_rows.append(metric_row("② 日周期 lag24", metrics(te["y"], te["lag24"])))

base_rows.append(metric_row("③ 周周期 lag168", metrics(te["y"], te["lag168"])))
base_rows.append(metric_row("④ 周周期×水平比", metrics(te["y"], te["lag168"] * lvl)))

base_tbl = pd.DataFrame(base_rows)
print(base_tbl.to_string(index=False))
print("日周期 / 周周期 MAE 比 %.4f" % (base_tbl["MAE"].iloc[1] / base_tbl["MAE"].iloc[2]))

assert abs(float(base_tbl["MAE"].iloc[0]) - 30.2482) < 1e-3
assert abs(float(base_tbl["MAE"].iloc[1]) - 21.5898) < 1e-3
assert abs(float(base_tbl["MAE"].iloc[2]) - 19.1805) < 1e-3
assert base_tbl["MAE"].iloc[2] < base_tbl["MAE"].iloc[1]        # 周周期赢日周期
assert base_tbl["MAE"].iloc[2] < base_tbl["MAE"].iloc[3]        # 水平比反而帮了倒忙

## 任务 5：Ridge 与 LightGBM（§五）

In [ ]:
scaler = StandardScaler().fit(tr[COLS])
ridge = Ridge(alpha=1.0).fit(scaler.transform(tr[COLS]), tr["y"].to_numpy(float))
pred_ridge = ridge.predict(scaler.transform(te[COLS]))

gbm = fit_lgb(tr[COLS], tr["y"].to_numpy(float))
pred_gbm = gbm.predict(te[COLS])

lgb_tbl = pd.DataFrame([
    metric_row("⑤ Ridge(α=1)", metrics(te["y"], pred_ridge)),
    metric_row("⑥ LightGBM", metrics(te["y"], pred_gbm)),
])
leader = pd.concat([base_tbl, lgb_tbl], ignore_index=True)
print(leader.to_string(index=False))

imp = pd.Series(gbm.feature_importances_, index=COLS).sort_values(ascending=False)
print("特征重要性 top8", {k: int(v) for k, v in imp.head(8).items()})
print("重要性最低 5 个", {k: int(v) for k, v in imp.tail(5).items()})

assert abs(float(leader["MAE"].iloc[4]) - 14.2251) < 1e-2
assert abs(float(leader["MAE"].iloc[5]) - 12.6689) < 1e-2
assert imp.index[0] == "lag24"                     # 最重要的居然是「昨天同一时刻」
assert float(leader["R2"].iloc[5]) > 0.93
assert float(leader["MAE"].iloc[5]) < float(leader["MAE"].iloc[4])

## 任务 6：误差的时段结构（§六）

In [ ]:
res_gbm = te["y"].to_numpy(float) - pred_gbm

seg = pd.DataFrame({
    "gbm": np.abs(res_gbm),
    "lag24": np.abs(te["y"] - te["lag24"]),
    "lag168": np.abs(te["y"] - te["lag168"]),
}).groupby(te.index.hour).mean()
print(seg.round(4).to_string())

SEGMENTS = {"夜间": (0, 5), "早峰": (6, 9), "日间": (10, 16), "晚峰": (17, 21), "深夜": (22, 23)}
seg_tbl = pd.DataFrame([seg_mean_row(seg, nm, lo, hi) for nm, (lo, hi) in SEGMENTS.items()])
print(seg_tbl.to_string(index=False))

mon_tbl = pd.DataFrame({"m": te.index.month, "gbm": np.abs(res_gbm)}).groupby("m").mean()
print("分月 MAE", {int(k): round(float(v), 4) for k, v in mon_tbl["gbm"].items()})

assert abs(float(seg.loc[17:21, "gbm"].mean()) - 16.7882) < 1e-3
assert abs(float(seg.loc[0:5, "gbm"].mean()) - 11.3963) < 1e-3
assert seg.loc[17:21, "gbm"].mean() > seg.loc[10:16, "gbm"].mean()      # 晚峰最难
assert seg.loc[17:21, "gbm"].mean() < seg.loc[17:21, "lag24"].mean()    # 但仍显著优于基线
assert len(seg_tbl) == 5

## 任务 7：直接法多步预测（§七）

In [ ]:
tcal = cal_block(y.index, "tgt_")        # 目标时刻的日历（列名带 tgt_ 前缀）
pos = {q: i for i, q in enumerate(y.index)}
tr_pos = np.array([pos[q] for q in tr.index])
te_pos = np.array([pos[q] for q in te.index])
yv = y.to_numpy(float)
tv = t.to_numpy(float)

d0 = np.array([i for i in range(len(yv)) if y.index[i].hour == 0])
opos = np.array([i for i in d0 if i >= te_pos[0] and i + H <= te_pos[-1]])
truth = np.stack([yv[opos + h] for h in range(1, H + 1)], axis=1)
print("发布日数", len(opos), "|", y.index[opos[0]], "~", y.index[opos[-1]])
print("真值矩阵 shape", truth.shape, "（= 发布日数 × 视野）")

# ---- 直接法：为 h=1..24 各训一个模型 ----
direct_models = {}
for h in range(1, H + 1):
    ok = tr_pos + h <= tr_pos[-1]
    X = pd.concat([tr[ok].reset_index(drop=True)[COLS],
                   tcal.iloc[tr_pos[ok] + h].reset_index(drop=True)], axis=1)
    direct_models[h] = fit_lgb(X, yv[tr_pos[ok] + h])
print("已训练", len(direct_models), "个模型")

mask = np.isin(te_pos, opos)
pred_direct = np.empty((len(opos), H))
for h in range(1, H + 1):
    XX = pd.concat([te[mask].reset_index(drop=True)[COLS],
                    tcal.iloc[opos + h].reset_index(drop=True)], axis=1)
    pred_direct[:, h - 1] = direct_models[h].predict(XX)

truth_naive = np.stack([yv[opos + h - 24] for h in range(1, H + 1)], axis=1)
mae_direct = np.abs(truth - pred_direct).mean(axis=0)
mae_naive = np.abs(truth - truth_naive).mean(axis=0)
print("直接法 日均 MAE %.4f | 日周期基线 %.4f" % (mae_direct.mean(), mae_naive.mean()))
print("h=1 %.4f → h=24 %.4f（×%.3f）"
      % (mae_direct[0], mae_direct[-1], mae_direct[-1] / mae_direct[0]))

assert len(opos) == 70
assert truth.shape == (70, 24)
assert abs(float(mae_direct.mean()) - 16.1107) < 1e-2
assert abs(float(mae_direct[0]) - 12.0749) < 1e-2
assert abs(float(mae_direct[-1]) - 13.6702) < 1e-2
assert float(mae_direct.mean()) < float(mae_naive.mean())

## 任务 8：递归法与 teacher forcing 上界（§八）

In [ ]:
# 只训一个「下一小时」模型：足够 168 小时历史起点才算样本
ok_rec = tr_pos - 1 >= N_HEAD
rec = fit_lgb(reduced_matrix(tr_pos[ok_rec], yv, tcal, tv), yv[tr_pos[ok_rec]])
print("递归法训练样本", int(ok_rec.sum()))

p_tf = rec.predict(teacher_forcing_rows(opos, yv, tcal, tv)).reshape(len(opos), H)

# ---- 递归法：每预测一格就把结果接回历史 ----
pred_rec = np.empty((len(opos), H))
for i, p in enumerate(opos):
    ext = list(yv[:p + 1])
    for k in range(1, H + 1):
        pred_rec[i, k - 1] = float(rec.predict(pd.DataFrame(
            [red_row(p + k, np.asarray(ext), tcal, tv)], columns=RED_COLS))[0])
        ext.append(pred_rec[i, k - 1])

mae_tf = np.abs(truth - p_tf).mean(axis=0)
mae_rec = np.abs(truth - pred_rec).mean(axis=0)
print("teacher forcing 日均 MAE %.4f" % mae_tf.mean())
print("递归法           日均 MAE %.4f" % mae_rec.mean())
print("递归法 h=1 %.4f → h=24 %.4f（×%.3f）"
      % (mae_rec[0], mae_rec[-1], mae_rec[-1] / mae_rec[0]))

assert abs(float(mae_tf.mean()) - 13.0980) < 1e-2
assert abs(float(mae_rec.mean()) - 13.1118) < 1e-2
assert float(mae_rec.mean()) < float(mae_direct.mean())     # 反直觉：递归反而更好

## 任务 9：归因拆解（§九）

In [ ]:
gap_feature = float(mae_tf.mean() - mae_direct.mean())
gap_drift = float(mae_rec.mean() - mae_tf.mean())
print("特征对齐带来的收益 %+.4f kW | 递归的误差累积 %+.4f kW" % (gap_feature, gap_drift))
print("误差累积占两者之和的 %.1f%%" % (100 * gap_drift / (gap_feature + gap_drift)))

print("三类多步特征的口径对照：")
print("  ❶ 日周期基线（把昨天的 24h 直接平移一天）")
print("  ❷ 直接法（24 个模型，特征全部锁死在原点 t）")
print("  ❸ 精简特征 + 真值历史（上界诊断，不可上线）")
print("  ❹ 精简特征 + 递归历史（可上线）")

step_tbl = pd.DataFrame({"h": np.arange(1, H + 1),
                         "❶日周期基线": np.round(mae_naive, 4),
                         "❷直接法": np.round(mae_direct, 4),
                         "❸精简+真值": np.round(mae_tf, 4),
                         "❹精简+递归": np.round(mae_rec, 4)})
print(step_tbl.to_string(index=False))

assert abs(gap_feature + 3.0127) < 1e-2          # 负值：精简特征反而更好
assert abs(gap_drift - 0.0138) < 1e-2            # 误差累积几乎为零
assert abs(gap_drift / (gap_feature + gap_drift)) < 0.05
assert float(mae_naive[-1] / mae_naive[0]) > 1.2         # 基线越往后越差
assert float(mae_rec[-1] / mae_rec[0]) < 1.2             # 递归法几乎不随 h 变差

## 任务 10：LSTM 对照实验（§十）

In [ ]:
pred_raw, l0_raw, l1_raw, nparam = train_lstm(
    yv, tv, FEAT["h_sin"].to_numpy(float), FEAT["h_cos"].to_numpy(float),
    cut + N_HEAD, opos, 0.0, 1.0, 0.0, 1.0)

pred_norm, l0_norm, l1_norm, _ = train_lstm(
    yv, tv, FEAT["h_sin"].to_numpy(float), FEAT["h_cos"].to_numpy(float),
    cut + N_HEAD, opos,
    float(yv[:cut + N_HEAD].mean()), float(yv[:cut + N_HEAD].std()),
    float(tv[:cut + N_HEAD].mean()), float(tv[:cut + N_HEAD].std()))

mae_raw = float(np.abs(truth - pred_raw).mean())
mae_norm = float(np.abs(truth - pred_norm).mean())
print("参数量", nparam, "| 训练样本", cut + N_HEAD - N_HEAD - H + 1)
print("不归一化：loss %.5f → %.5f | 日均 MAE %.4f" % (l0_raw, l1_raw, mae_raw))
print("归一化　：loss %.5f → %.5f | 日均 MAE %.4f" % (l0_norm, l1_norm, mae_norm))
print("归一化把误差压低了 %.1f 倍" % (mae_raw / mae_norm))
print("但仍打不过 LightGBM（%.4f）与递归法（%.4f）" % (float(leader["MAE"].iloc[5]), mae_rec.mean()))

assert abs(mae_raw - 409.7550) < 0.5
assert abs(mae_norm - 17.4668) < 0.5
assert mae_raw / mae_norm > 20                 # 归一化与否差了一个数量级
assert mae_norm > float(mae_rec.mean())        # 归一化之后仍然输

## 任务 11：残差白噪声检验（§十一）

In [ ]:
lb = acorr_ljungbox(res_gbm, lags=[24, 48, 72], return_df=True)
res_acf = {k: round(float(pd.Series(res_gbm).autocorr(k)), 6) for k in (1, 24, 168)}
print("Ljung-Box p 值", dict(zip([24, 48, 72], np.round(lb["lb_pvalue"].to_numpy(), 8))))
print("残差 ACF", res_acf)
print("残差 std / y std = %.4f" % (res_gbm.std() / te["y"].std()))

assert bool((lb["lb_pvalue"].to_numpy() < 0.01).all())     # 强烈拒绝白噪声
assert abs(res_acf[1] - (-0.055219)) < 1e-5
assert abs(float(res_gbm.std() / te["y"].std()) - 0.2456) < 1e-3
assert abs(res_acf[168]) < abs(res_acf[24])               # 周周期残差比日周期小

## 任务 12：画三张图（§十二）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

# 最后 7 个发布日拼成一个连续 168 小时窗口：起点 = 第 7 个发布日的下一小时
i0 = int(opos[-7] + 1)
xidx = y.index[i0:i0 + 168]

ax = axes[0]
ax.plot(xidx, yv[i0:i0 + 168], label="真值", lw=1.8, color="#1f77b4")
ax.plot(xidx, pred_direct[-7:].ravel(), label="多步预测(7天×24)", lw=1.3, color="#d62728")
ax.plot(xidx, yv[i0 - 24:i0 - 24 + 168], label="日周期基线", lw=1.0, ls="--", color="#7f7f7f")
ax.set_title("最后 7 天：日前 24 步预测")
ax.set_ylabel("负荷 / kW")
ax.legend(fontsize=8)
ax.tick_params(axis="x", rotation=30)

ax = axes[1]
ax.plot(seg.index, seg["gbm"], marker="o", label="LightGBM", color="#2ca02c")
ax.plot(seg.index, seg["lag24"], marker="s", ls="--", label="日周期基线", color="#7f7f7f")
ax.plot(seg.index, seg["lag168"], marker="^", ls=":", label="周周期基线", color="#ff7f0e")
ax.set_title("误差的时段结构")
ax.set_xlabel("小时")
ax.set_ylabel("MAE / kW")
ax.legend(fontsize=8)

ax = axes[2]
top = imp.head(15)[::-1]
ax.barh(top.index, top.to_numpy(), color="#9467bd")
ax.set_title("特征重要性 top15")
ax.set_xlabel("split 次数")

plt.tight_layout()
print("三张图已生成")

assert len(axes) == 3
assert imp.head(15).index[0] == "lag24"
assert seg["gbm"].idxmax() in (19, 20)

## 任务 13：自动生成结题报告（§十三）

In [ ]:
best_name = leader.loc[leader["MAE"].idxmin(), "模型"]
best_mae = float(leader["MAE"].min())
worst_name = leader.loc[leader["MAE"].idxmax(), "模型"]
gain_vs_naive = float(mae_naive.mean()) / float(mae_direct.mean())

REPORT = (
    "# 台区负荷日前预测 · 结题报告\n\n"
    f"- 数据：{STATION}，{len(s_raw)} 小时规则序列，清洗后可用样本 {n} 个\n"
    f"- 切分：训练 {len(tr)} / 测试 {len(te)}（按时间 8:2，"
    f"测试段 {te.index.min():%Y-%m-%d} ~ {te.index.max():%Y-%m-%d}）\n"
    f"- 最佳单步模型：{best_name}，MAE = {best_mae:.4f} kW，"
    f"MAPE = {float(leader['MAPE'].min()):.4f}%，R2 = {float(leader['R2'].max()):.6f}\n"
    f"- 最弱模型：{worst_name}，MAE = {float(leader['MAE'].max()):.4f} kW\n"
    f"- 日前 24 步（每天 0 点发布，共 {len(opos)} 天）："
    f"直接法 {float(mae_direct.mean()):.4f} kW / 递归法 {float(mae_rec.mean()):.4f} kW / "
    f"日周期基线 {float(mae_naive.mean()):.4f} kW，提升 {gain_vs_naive:.2f} 倍\n"
    f"- 归因：递归的误差累积仅 {gap_drift:+.4f} kW，"
    f"真正拉开差距的是特征对齐（{gap_feature:+.4f} kW）\n"
    f"- 环境：lightgbm {lgb.__version__}，OMP_NUM_THREADS={os.environ['OMP_NUM_THREADS']}\n"
)
print(REPORT)

assert "最佳单步模型" in REPORT
assert best_name == "⑥ LightGBM"
assert "12.6689" in REPORT
assert "16.1107" in REPORT
assert "13.1118" in REPORT
assert REPORT.count("\n") >= 6

## 任务 14：跨台区泛化（§十四）

In [ ]:
one_b = df[df["台区编号"] == OTHER].set_index("时间戳").sort_index()
y_b = clean_by_hour(one_b["负荷值"].asfreq("h"))
t_b = one_b["温度"].asfreq("h").interpolate(limit_direction="both")

FB = make_features(y_b, t_b).dropna()
COLS_B = [c for c in FB.columns if c != "y"]
cut_b = int(len(FB) * 0.8)
tr_b, te_b = FB.iloc[:cut_b], FB.iloc[cut_b:]

gbm_b = fit_lgb(tr_b[COLS_B], tr_b["y"].to_numpy(float))
pred_b = gbm_b.predict(te_b[COLS_B])

b_tbl = pd.DataFrame([
    metric_row("① 持续性 lag1", metrics(te_b["y"], te_b["lag1"])),
    metric_row("② 日周期 lag24", metrics(te_b["y"], te_b["lag24"])),
    metric_row("③ 周周期 lag168", metrics(te_b["y"], te_b["lag168"])),
    metric_row("⑥ LightGBM", metrics(te_b["y"], pred_b)),
])
print("B 台区（工业型，日间平台 + 周末腰斩）")
print(b_tbl.to_string(index=False))

seg_b = pd.DataFrame({"gbm": np.abs(te_b["y"].to_numpy(float) - pred_b),
                      "lag24": np.abs(te_b["y"] - te_b["lag24"])}).groupby(te_b.index.hour).mean()
print("B 台区 晚峰(17-21) MAE %.4f | 日间(10-16) MAE %.4f"
      % (seg_b.loc[17:21, "gbm"].mean(), seg_b.loc[10:16, "gbm"].mean()))

assert abs(float(b_tbl["MAE"].iloc[1]) - 71.3448) < 1e-2    # 日周期基线在 B 上彻底崩了
assert abs(float(b_tbl["MAE"].iloc[2]) - 17.1623) < 1e-2    # 周周期基线反而很强
assert abs(float(b_tbl["MAE"].iloc[3]) - 13.1559) < 1e-2
assert float(b_tbl["MAE"].iloc[3]) < float(b_tbl["MAE"].iloc[1])
assert seg_b.loc[10:16, "gbm"].mean() > seg_b.loc[17:21, "gbm"].mean()   # A 与 B 的难点时段正好相反

## 自查清单（能不看笔记答出来才算过）

- [ ] 14 个任务的所有 assert 全部通过
- [ ] `drop_duplicates` 必须在哪一步之前？不这样做会怎样？
- [ ] 清洗阈值为什么必须「按小时分组」？换成 B 台区会发生什么？
- [ ] 本数据里 `9999` 这个值代表什么？它在 7 个越界点里占了几个？
- [ ] 温度用 `|T−22|` 而不是 `T`，相关系数从多少涨到多少？
- [ ] `rolling(168)` 为什么会让表头 168 行全变 NaN？8592 这个数字怎么来的？
- [ ] 泄漏自检里 `ma24` 的窗口实际覆盖 `y` 的哪一段？为什么不是 `y[j-23:j+1]`？
- [ ] 为什么周周期基线（19.1805）比日周期基线（21.5898）好？
- [ ] 「周周期 × 水平比」为什么反而更差？这说明加修正项之前该先量什么？
- [ ] 直接法的训练样本为什么要卡 `tr_pos + h <= tr_pos[-1]`？
- [ ] 递归法里 `yv[:p+1]` 写成 `yv[:p]` 会让 MAE 从多少涨到多少？
- [ ] teacher forcing 为什么不能上线？它的作用是什么？
- [ ] 「特征对齐收益」与「误差累积」各是多少？谁在主导？
- [ ] 为什么这里的递归误差累积几乎为零？把 horizon 提到 200 小时会怎样？
- [ ] LSTM 不归一化时 loss 从 451 降到多少？归一化后是 0.287 降到多少？
- [ ] 为什么 LSTM 调参（17.4668 → 19.2211）反而更差？
- [ ] 残差 `ACF(24) = −0.111` 是「漏抓」还是「过冲」？该怎么修？
- [ ] A 台区和 B 台区「最难的时段」分别是什么？为什么相反？
- [ ] B 台区的日周期基线 MAE 是多少？为什么这么差？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 原始行数 / 去重后 / 重复行 | 17526 / 17520 / 6 |
| A 台区缺失（负荷 / 温度） | 70 / 15 |
| 越界点数（其中负值） | 7 / 2 |
| 清洗前 std / 清洗后 std | 129.2674 / 72.1933 |
| 最大单点修正量 | 9480.56 kW |
| 特征表形状 / 含 NaN 行 | (8760, 25) / 168 |
| 节假日小时 / 周末小时 | 672 / 2496 |
| 23↔0 编码弦长 | 0.261052 |
| 温度线性 / V 形相关 | −0.188929 / 0.406437 |
| 样本 / 训练 / 测试 | 8592 / 6873 / 1719 |
| 测试段 y 均值 | 483.9447 |
| ① 持续性 `lag1` MAE | 30.2482 |
| ② 日周期 `lag24` MAE | 21.5898 |
| ③ 周周期 `lag168` MAE | 19.1805 |
| ④ 周周期×水平比 MAE | 21.6910 |
| ⑤ Ridge(α=1) MAE | 14.2251 |
| ⑥ LightGBM MAE / R² | 12.6689 / 0.9386 |
| 特征重要性 top1 / top2 | `lag24` 692 / `lag1` 674 |
| 分时段 MAE：夜间 / 日间 / 晚峰 | 11.3963 / 11.2119 / 16.7882 |
| 分月 MAE：10 / 11 / 12 月 | 12.1156 / 12.5286 / 12.9943 |
| 发布日数 / 真值矩阵 | 70 / (70, 24) |
| ❷ 直接法 日均 MAE | 16.1107 |
| ❹ 递归法 日均 MAE | 13.1118 |
| ❶ 日周期基线 日均 MAE | 21.6100 |
| ❸ teacher forcing 日均 MAE | 13.0980 |
| 特征对齐收益 / 误差累积 | −3.0127 / +0.0138 |
| 递归 off-by-one（`yv[:p]`）MAE | 21.1492 |
| LSTM 不归一化 MAE | 409.7550 |
| LSTM 归一化 MAE | 17.4668 |
| LSTM 归一化前后倍数 | 23.5 |
| 残差 Ljung-Box p（24/48/72） | 3.17e-06 / 6e-08 / 6.93e-06 |
| 残差 ACF(1) / ACF(24) / ACF(168) | −0.055219 / −0.111229 / +0.032333 |
| 残差 std / y std | 0.2456 |
| B 台区 日周期 / 周周期 MAE | 71.3448 / 17.1623 |
| B 台区 LightGBM MAE | 13.1559 |
| B 台区 日间 / 晚峰 MAE | 21.7747 / 12.2618 |